In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType

In [2]:
spark = (
    SparkSession.builder
    .appName("yahoo_finance")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

In [3]:
schema = StructType() \
    .add("symbol", StringType()) \
    .add("date", StringType()) \
    .add("price", DoubleType()) \
    .add("return", DoubleType()) \
    .add("volatility", DoubleType())

In [4]:
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "yahoo_finance") \
    .option("startingOffsets", "latest") \
    .load()

In [8]:
df_final = df_kafka.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

In [9]:
spark._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://minio:9000")
spark._jsc.hadoopConfiguration().set("fs.s3a.access.key", "minio")
spark._jsc.hadoopConfiguration().set("fs.s3a.secret.key", "minio123")
spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
spark._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

In [10]:
df_final.writeStream \
    .format("parquet") \
    .option("path", "s3a://data-lake/curated/yahoo_finance") \
    .option("checkpointLocation", "/tmp/stock_cp") \
    .outputMode("append") \
    .start()

In [11]:
df_final

DataFrame[symbol: string, date: string, price: double, return: double, volatility: double]